In [1]:
# !pip install numpy==1.26.4
# !pip install shap --no-deps
# !pip install lime --no-deps
# !pip install lime --no-deps
# !pip install alibi --no-deps

In [2]:
# =======================
# 📦 IMPORTACIONES
# =======================

import sys
from pathlib import Path

ROOT = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT))


# Built-in
import os
import sys
import re
import time
import json
import random
import warnings
from typing import List, Tuple, Dict
import operator
import re
from pathlib import Path
import numpy as np
import pandas as pd

np.random.seed(42)
random.seed(42)

# NumPy, Pandas, Matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    log_loss, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, pairwise_distances
)
from sklearn.exceptions import NotFittedError
from collections import defaultdict
from sklearn.metrics import classification_report, confusion_matrix


# Flower
from flwr.client import ClientApp, NumPyClient
from flwr.common import (
    Context, NDArrays, Metrics, Scalar,
    ndarrays_to_parameters, parameters_to_ndarrays
)
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# LORE
from lore_sa.dataset import TabularDataset
from lore_sa.bbox import sklearn_classifier_bbox
from lore_sa.encoder_decoder import ColumnTransformerEnc
from lore_sa.lore import TabularGeneticGeneratorLore
from lore_sa.surrogate.decision_tree import SuperTree
from lore_sa.rule import Expression, Rule

from lore_sa.client_utils import ClientUtilsMixin

# Otros
from pathlib import Path
from filelock import FileLock  # pip install filelock
import pandas as pd, os
from graphviz import Digraph
from tqdm import tqdm
from datetime import datetime
import cProfile, pstats, io
from flwr_datasets.partitioner import IidPartitioner, DirichletPartitioner
import copy
from sklearn.model_selection import train_test_split
import shutil
from lore_sa.client_utils.explanation_intersection import ExplanationIntersection

import shap
from lime.lime_tabular import LimeTabularExplainer
from alibi.explainers import AnchorTabular

from lore_sa.client_utils.cumple_regla import ReglaEvaluator
from lore_sa.client_utils.explanation_metrics import Explainer_metrics



c:\Users\pablo\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\pablo\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
2026-03-24 19:48:22,216	INFO util.py:154 -- Outdated packages:
  ipywidgets==7.8.1 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-24 19:48:27,857 graphviz._tools DEBUG    deprecate positional args: graphviz.backend.piping.pipe(['renderer', 'formatter', 'neato_no_op', 'quiet'])
2026-03-24 19:48:27,860 graphviz._tools DEBUG    deprecate positional args: graphviz.backend.rendering.render(['renderer', 'formatter', 'neato_no_op', 'quiet'])
2026-03-24 19

2026-03-24 19:48:30,508 tensorflow   WARNING  From c:\Users\pablo\anaconda3\Lib\site-packages\alibi\utils\tf.py:11: The name tf.Session is deprecated. Please use tf.compat.v1.Session instead.



In [3]:
# =======================
# ⚙️ VARIABLES GLOBALES
# =======================



UNIQUE_LABELS = []
FEATURES = []

NUM_TRAIN_ROUNDS = 2        # rondas donde entrenas la NN
NUM_SERVER_ROUNDS = 3       # la última solo para explicaciones
NUM_CLIENTS = 2
SEED = 42

NON_IID = False   # o False para los experimentos IID
NON_IID_ALPHA = 0.5  # por ejemplo, Dirichlet más sesgado

MIN_AVAILABLE_CLIENTS = NUM_CLIENTS
fds = None  # Cache del FederatedDataset
CAT_ENCODINGS = {}
USING_DATASET = None

GLOBAL_TEST_IDX = None
GLOBAL_TEST_HASHES = None


# ==============================================
# 🧹 Borrar TODOS los CSV individuales de clientes
# ==============================================

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Borra TODO lo que haya dentro (csv, pth, imágenes, etc.)
for f in results_dir.iterdir():
    if f.is_file():
        try:
            f.unlink()
        except Exception:
            pass




# =======================
# 🔧 UTILIDADES MODELO
# =======================

def get_model_parameters(tree_model, nn_model):
    tree_params = [
        int(tree_model.get_params()["max_depth"] or -1),
        int(tree_model.get_params()["min_samples_split"]),
        int(tree_model.get_params()["min_samples_leaf"]),
    ]
    nn_weights = [v.cpu().detach().numpy() for v in nn_model.state_dict().values()]
    return {
        "tree": tree_params,
        "nn": nn_weights,
    }


def set_model_params(tree_model, nn_model, params):
    tree_params = params["tree"]
    nn_weights = params["nn"]

    # Solo si tree_model no es None y tiene set_params
    if tree_model is not None and hasattr(tree_model, "set_params"):
        max_depth = tree_params[0] if tree_params[0] > 0 else None
        tree_model.set_params(
            max_depth=max_depth,
            min_samples_split=tree_params[1],
            min_samples_leaf=tree_params[2],
        )

    # Actualizar pesos de la red neuronal
    state_dict = nn_model.state_dict()
    for (key, _), val in zip(state_dict.items(), nn_weights):
        state_dict[key] = torch.tensor(val)
    nn_model.load_state_dict(state_dict)





# =======================
# 📥 PREPROCESADO DATASET
# =======================
def preprocess_df(df: pd.DataFrame, dataset_name: str, class_col: str) -> pd.DataFrame:
    df = df.copy()

    if "adult" in dataset_name.lower():
        df.drop(columns=['fnlwgt', 'education-num', 'capital-gain', 'capital-loss'],
                inplace=True, errors="ignore")

    elif "churn" in dataset_name.lower():
        df.drop(columns=['customerID', 'TotalCharges'],
                inplace=True, errors="ignore")
        if "MonthlyCharges" in df.columns:
            df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
        if "tenure" in df.columns:
            df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce')
        if "SeniorCitizen" in df.columns:
            df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'}).astype(str)
        df.dropna(subset=[c for c in ["MonthlyCharges", "tenure"] if c in df.columns], inplace=True)

    elif "breastcancer" in dataset_name.lower():
        df.drop(columns=['id'], inplace=True, errors='ignore')

    # object -> category (solo baja cardinalidad)
    for col in df.select_dtypes(include=["object"]).columns:
        if col != class_col and df[col].nunique(dropna=True) < 50:
            df[col] = df[col].astype("category")

    return df


def _stable_row_hash(df: pd.DataFrame) -> np.ndarray:
    return pd.util.hash_pandas_object(df, index=False).astype("uint64").to_numpy()

# =======================
# 📥 CARGAR DATOS
# =======================

def get_global_onehot_info(flower_dataset_name: str, class_col: str):
    """
    Lee TODO el pool (train con num_partitions=1) para fijar:
    - cat_features (categorical cols)
    - num_features
    - categories_global (OHE categories_ en el orden de cat_features)
    - onehot_columns (nombres finales onehot)
    """
    fds_tmp = FederatedDataset(
        dataset=flower_dataset_name,
        partitioners={"train": IidPartitioner(num_partitions=1)}
    )
    df_all = fds_tmp.load_partition(0, "train").with_format("pandas")[:]
    df_all = preprocess_df(df_all, flower_dataset_name, class_col)

    # asegurar category dtype
    for col in df_all.select_dtypes(include=["object"]).columns:
        if col != class_col and df_all[col].nunique(dropna=True) < 50:
            df_all[col] = df_all[col].astype("category")

    cat_features = [c for c in df_all.columns if df_all[c].dtype.name == "category" and c != class_col]
    num_features = [c for c in df_all.columns if df_all[c].dtype.kind in "fi" and c != class_col]

    if len(cat_features) > 0:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        ohe.fit(df_all[cat_features])
        categories_global = ohe.categories_
        onehot_columns = ohe.get_feature_names_out(cat_features).tolist()
    else:
        categories_global = []
        onehot_columns = []

    return cat_features, num_features, categories_global, onehot_columns, df_all




# ============================================
# 📥 CARGA GENERAL + TEST GLOBAL SIN FUGA
# ============================================
def load_data_general(flower_dataset_name: str, class_col: str, partition_id: int, num_partitions: int):
    """
    Devuelve:
      X_train, y_train,
      X_test_local, y_test_local,
      X_test_global, y_test_global,
      tabular_dataset, feature_names_out, label_encoder,
      num_transformer, numeric_features,
      encoder (ColumnTransformerEnc), preprocessor
    """
    global fds, UNIQUE_LABELS, FEATURES
    global GLOBAL_TEST_IDX, GLOBAL_TEST_HASHES

    # 1) Info global OHE + df_all pool
    cat_features, num_features, categories_global, onehot_columns, df_all = get_global_onehot_info(
        flower_dataset_name, class_col
    )

    # 2) LabelEncoder global (clases estables)
    if not UNIQUE_LABELS:
        le_global = LabelEncoder()
        le_global.fit(df_all[class_col])
        UNIQUE_LABELS[:] = le_global.classes_.tolist()

    label_encoder = LabelEncoder()
    label_encoder.classes_ = np.array(UNIQUE_LABELS)

    y_all = label_encoder.transform(df_all[class_col])

    # 3) Definir TEST GLOBAL una sola vez (idx + hashes estables)
    if GLOBAL_TEST_IDX is None:
        idx = np.arange(len(df_all))
        _, GLOBAL_TEST_IDX = train_test_split(
            idx,
            test_size=0.2,
            random_state=SEED,
            stratify=y_all if len(np.unique(y_all)) > 1 else None
        )
        row_hash_all = _stable_row_hash(df_all)
        GLOBAL_TEST_HASHES = set(row_hash_all[GLOBAL_TEST_IDX].tolist())

    # 4) Crear/usar FederatedDataset particionado (por filas)
    if fds is None:
        if NON_IID:
            partitioner = DirichletPartitioner(
                num_partitions=num_partitions,
                alpha=NON_IID_ALPHA,
                partition_by=class_col,
            )
        else:
            partitioner = IidPartitioner(num_partitions=num_partitions)

        fds = FederatedDataset(
            dataset=flower_dataset_name,
            partitioners={"train": partitioner},
        )

    df_client = fds.load_partition(partition_id, "train").with_format("pandas")[:]
    df_client = preprocess_df(df_client, flower_dataset_name, class_col)

    # 5) Eliminar filas que estén en el TEST GLOBAL (sin fuga)
    row_hash_client = _stable_row_hash(df_client)
    keep_mask = ~np.isin(row_hash_client, np.fromiter(GLOBAL_TEST_HASHES, dtype="uint64"))
    df_client = df_client.loc[keep_mask].copy()

    # 6) TabularDataset/descriptor (cliente) para LORE
    tabular_dataset = TabularDataset(df_client.copy(), class_name=class_col)
    descriptor = tabular_dataset.descriptor

    # Asegurar distinct_values local (por si viene vacío)
    for col, info in descriptor.get("categorical", {}).items():
        if "distinct_values" not in info or not info["distinct_values"]:
            info["distinct_values"] = list(df_client[col].dropna().unique())

    # 7) X/y cliente (sin onehot aún)
    y = label_encoder.transform(df_client[class_col])
    X_raw = df_client.drop(columns=[class_col])

    numeric_features = list(descriptor.get("numeric", {}).keys())
    categorical_features = list(descriptor.get("categorical", {}).keys())

    # Ojo: aquí mantenemos el mismo orden que usa el descriptor del cliente
    FEATURES[:] = numeric_features + categorical_features

    num_idx = list(range(len(numeric_features)))
    cat_idx = list(range(len(numeric_features), len(FEATURES)))

    # 8) Preprocessor con categorías globales (dim estable)
    transformers = [("num", "passthrough", num_idx)]
    if len(categorical_features) > 0:
        # IMPORTANTÍSIMO: categories_global está en el orden de cat_features (global)
        # pero aquí categorical_features puede venir en otro orden -> reordenamos categories_global
        cat_to_pos = {c: i for i, c in enumerate(cat_features)}
        cats_ordered = [categories_global[cat_to_pos[c]] for c in categorical_features]

        transformers.append((
            "cat",
            OneHotEncoder(
                sparse_output=False,
                handle_unknown="ignore",
                categories=cats_ordered
            ),
            cat_idx
        ))

    preprocessor = ColumnTransformer(transformers)

    # 9) Split local + FIT SOLO con train local
    X_train_raw, X_test_local_raw, y_train, y_test_local = train_test_split(
        X_raw[FEATURES], y,
        test_size=0.3,
        random_state=SEED,
        stratify=y if len(np.unique(y)) > 1 else None
    )

    X_train = preprocessor.fit_transform(X_train_raw.to_numpy())
    X_test_local = preprocessor.transform(X_test_local_raw.to_numpy())

    # 10) Construir test global REAL (mismo preprocessor del cliente)
    df_global = df_all.iloc[GLOBAL_TEST_IDX].copy()
    df_global = preprocess_df(df_global, flower_dataset_name, class_col)

    X_test_global = preprocessor.transform(df_global.drop(columns=[class_col])[FEATURES].to_numpy())
    y_test_global = label_encoder.transform(df_global[class_col])

    # 11) Feature names finales (num + onehot) para NN/servidor
    feature_names_out = []
    feature_names_out += list(numeric_features)
    if len(categorical_features) > 0:
        cat_names = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features).tolist()
        feature_names_out += cat_names

    FEATURES[:] = feature_names_out  # ahora sí: columnas finales (onehot)

    # 12) Encoder LORE con distinct_values globales (para reglas legibles)
    descriptor_global = descriptor.copy()
    if "categorical" in descriptor_global and len(categorical_features) > 0:
        # cats_ordered ya está en orden de categorical_features
        for i, col in enumerate(categorical_features):
            if col in descriptor_global["categorical"]:
                descriptor_global["categorical"][col]["distinct_values"] = list(cats_ordered[i])

    encoder = ColumnTransformerEnc(descriptor_global)

    num_transformer = preprocessor.named_transformers_["num"] if "num" in preprocessor.named_transformers_ else None

    return (
        X_train, y_train,
        X_test_local, y_test_local,
        X_test_global, y_test_global,
        tabular_dataset, feature_names_out, label_encoder,
        num_transformer, numeric_features, encoder, preprocessor
    )


# =======================
# ✅ DATASET
# =======================
# DATASET_NAME = "pablopalacios23/adult"
# CLASS_COLUMN = "class"

# DATASET_NAME = "pablopalacios23/churn"
# CLASS_COLUMN = "Churn"

# DATASET_NAME = "pablopalacios23/HeartDisease"
# CLASS_COLUMN = "HeartDisease"

# DATASET_NAME = "pablopalacios23/breastcancer"
# CLASS_COLUMN = "diagnosis"

DATASET_NAME = "pablopalacios23/Diabetes"
CLASS_COLUMN = "Outcome"



# load_data_general(DATASET_NAME, CLASS_COLUMN, partition_id=0, num_partitions=NUM_CLIENTS)

### HOLDOUT DEL SERVIDOR

In [4]:
(X_train, y_train,
 X_test_local, y_test_local,
 X_test_global, y_test_global,
 dataset, feature_names, label_encoder,
 scaler, numeric_features, encoder, preprocessor) = load_data_general(
    DATASET_NAME, CLASS_COLUMN, partition_id=0, num_partitions=NUM_CLIENTS
)


print("\n📦 TRAIN (primeras filas):")
print(pd.DataFrame(X_train))

print(y_train)

print(X_train.shape)

print("\n🧪 TEST LOCAL (primeras filas):")
print(pd.DataFrame(X_test_local))

print(y_test_local)

print("\n🌍 TEST GLOBAL (primeras filas):")
print(pd.DataFrame(X_test_global))

print(y_test_global)

def overlap_check(A, B):
    hA = pd.util.hash_pandas_object(pd.DataFrame(A), index=False)
    hB = pd.util.hash_pandas_object(pd.DataFrame(B), index=False)
    return np.intersect1d(hA.values, hB.values).size


print("🔎 Overlap TRAIN vs TEST GLOBAL:",
      overlap_check(X_train, X_test_global))

print("🔎 Overlap TEST LOCAL vs TEST GLOBAL:",
      overlap_check(X_test_local, X_test_global))

print(feature_names)

2026-03-24 19:48:30,955 urllib3.connectionpool DEBUG    Starting new HTTPS connection (1): huggingface.co:443
2026-03-24 19:48:31,482 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /datasets/pablopalacios23/Diabetes/resolve/main/README.md HTTP/1.1" 404 0
2026-03-24 19:48:31,611 urllib3.connectionpool DEBUG    https://huggingface.co:443 "GET /api/datasets/pablopalacios23/Diabetes HTTP/1.1" 200 556
2026-03-24 19:48:31,746 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /datasets/pablopalacios23/Diabetes/resolve/90286a808fa902822a230872737f66665a681328/Diabetes.py HTTP/1.1" 404 0
2026-03-24 19:48:31,751 urllib3.connectionpool DEBUG    Starting new HTTPS connection (1): s3.amazonaws.com:443
2026-03-24 19:48:32,368 urllib3.connectionpool DEBUG    https://s3.amazonaws.com:443 "HEAD /datasets.huggingface.co/datasets/datasets/pablopalacios23/Diabetes/pablopalacios23/Diabetes.py HTTP/1.1" 404 0
2026-03-24 19:48:32,499 urllib3.connectionpool DEBUG    https://hu


📦 TRAIN (primeras filas):
       0      1      2     3      4     5      6     7
0    8.0  167.0  106.0  46.0  231.0  37.6  0.165  43.0
1    4.0  137.0   84.0   0.0    0.0  31.2  0.252  30.0
2    1.0  128.0   88.0  39.0  110.0  36.5  1.057  37.0
3    6.0  134.0   80.0  37.0  370.0  46.2  0.238  46.0
4    3.0  171.0   72.0  33.0  135.0  33.3  0.199  24.0
..   ...    ...    ...   ...    ...   ...    ...   ...
212  4.0  154.0   72.0  29.0  126.0  31.3  0.338  37.0
213  1.0   87.0   78.0  27.0   32.0  34.6  0.101  22.0
214  8.0  110.0   76.0   0.0    0.0  27.8  0.237  58.0
215  1.0  122.0   64.0  32.0  156.0  35.1  0.692  30.0
216  1.0  103.0   80.0  11.0   82.0  19.4  0.491  22.0

[217 rows x 8 columns]
[1 0 1 1 1 0 1 1 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0
 1 1 1 1 0 0 1 1 1 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0 1 0 0 0 1 0 0 0 1
 0 0 1 0 1 0 0 0 0 1 1 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 1 0 0 0 0 1
 0 0 0 0 1 0 0 0 0 0 1 0 1 1 0 1 0 1 0 0 1 0 1 1 0 0 0 1 1 0 0 0

# Cliente

In [5]:
from lore_sa.client_utils.client_RANDOMFOREST import make_client_app_rf

client_app = make_client_app_rf(
    dataset_name=DATASET_NAME,
    class_column=CLASS_COLUMN,
    num_clients=NUM_CLIENTS,
    num_train_rounds=NUM_TRAIN_ROUNDS,
    unique_labels=UNIQUE_LABELS,
    features=FEATURES,
    load_data_fn=load_data_general,
    set_params_fn=set_model_params,
    get_params_fn=get_model_parameters,
)

# Servidor

In [6]:
# ============================
# 📦 IMPORTACIONES NECESARIAS
# ============================
import os
import time
import json
import numpy as np
from typing import List, Tuple, Dict
from sklearn.tree import DecisionTreeClassifier

from flwr.common import Context, Metrics, Scalar, ndarrays_to_parameters
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg

from graphviz import Digraph
from lore_sa.surrogate.decision_tree import SuperTree

import torch
import torch.nn as nn
import torch.nn.functional as F




# ============================
# ⚙️ CONFIGURACIÓN GLOBAL
# ============================
# MIN_AVAILABLE_CLIENTS = 4
# NUM_SERVER_ROUNDS = 2

FEATURES = []  # se rellenan dinámicamente
UNIQUE_LABELS = []
LATEST_SUPERTREE_JSON = None
GLOBAL_MAPPING_JSON = None
FEATURE_NAMES_JSON = None
GLOBAL_SCALER_JSON = None


# ============================
# 🧠 UTILIDADES MODELO
# ============================
def create_model(input_dim, output_dim):
    from lore_sa.client_utils.client_RANDOMFOREST import Net    
    return Net(input_dim, output_dim)


def get_model_parameters(tree_model, nn_model):
    tree_params = [-1, 2, 1]
    nn_weights = [v.cpu().detach().numpy() for v in nn_model.state_dict().values()]
    return {
        "tree": tree_params,
        "nn": nn_weights,
    }

def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Dict[str, Scalar]:
    sums: Dict[str, float] = {}
    counts: Dict[str, int] = {}

    for n, met in metrics:
        for k, v in met.items():
            if isinstance(v, (float, int)):
                sums[k] = sums.get(k, 0.0) + n * float(v)
                counts[k] = counts.get(k, 0) + n

    return {k: sums[k] / counts[k] for k in sums}

# ============================
# 🚀 SERVIDOR FLOWER
# ============================

def server_fn(context: Context) -> ServerAppComponents:
    global FEATURES, UNIQUE_LABELS

    # Justo antes de llamar a create_model
    if not FEATURES or not UNIQUE_LABELS:
        
        load_data_general(DATASET_NAME, CLASS_COLUMN, partition_id=0, num_partitions=NUM_CLIENTS)


    FEATURES = FEATURES or ["feat_0", "feat_1"]  # fallback por si no se cargó antes
    UNIQUE_LABELS = UNIQUE_LABELS or ["Class_0", "Class_1"]


    model = create_model(len(FEATURES), len(UNIQUE_LABELS))
    initial_params = ndarrays_to_parameters(get_model_parameters(None, model)["nn"])

    strategy = FedAvg(
        min_available_clients=MIN_AVAILABLE_CLIENTS,
        fit_metrics_aggregation_fn=weighted_average,
        evaluate_metrics_aggregation_fn=weighted_average,
        initial_parameters=initial_params,
    )

    strategy.configure_fit = _inject_round(strategy.configure_fit)
    strategy.configure_evaluate = _inject_round(strategy.configure_evaluate)
    original_aggregate = strategy.aggregate_evaluate

    def custom_aggregate_evaluate(server_round, results, failures):
        global LATEST_SUPERTREE_JSON, GLOBAL_MAPPING_JSON, FEATURE_NAMES_JSON
        aggregated_metrics = original_aggregate(server_round, results, failures)

        # ============================
        # 🔹 Ronda final: NO fusionar nada
        # ============================
        if server_round > NUM_TRAIN_ROUNDS:
            return aggregated_metrics

        try:
            print(f"\n[SERVIDOR] 🌲 Generando SuperTree - Ronda {server_round}")
            from collections import defaultdict

            tree_nodes = []
            all_distincts = defaultdict(set)
            client_encoders = {}

            feature_names = None
            numeric_features = None
            class_names = None

            # 1) recolectar mapeos categóricos y metadatos
            for (_, evaluate_res) in results:
                metrics = evaluate_res.metrics
                # distinct_values_* para global_mapping
                for k, v in metrics.items():
                    if k.startswith("distinct_values_"):
                        cid = k.split("_")[-1]
                        enc = json.loads(v)
                        client_encoders[cid] = enc
                        for feat, d in enc.items():
                            all_distincts[feat].update(d["distinct_values"])

            global_mapping = {feat: sorted(list(vals)) for feat, vals in all_distincts.items()}

            # 2) recolectar árboles y demás metadatos por cliente
            for (_, evaluate_res) in results:
                metrics = evaluate_res.metrics
                for k, v in metrics.items():
                    if k.startswith("tree_ensemble_"):
                        cid = k.split("_")[-1]
                        trees_list = json.loads(v)

                        # lee estos una sola vez (son iguales por cliente)
                        if feature_names is None and f"encoded_feature_names_{cid}" in metrics:
                            feature_names = json.loads(metrics[f"encoded_feature_names_{cid}"])
                        if numeric_features is None and f"numeric_features_{cid}" in metrics:
                            numeric_features = json.loads(metrics[f"numeric_features_{cid}"])
                        if class_names is None and f"unique_labels_{cid}" in metrics:
                            class_names = json.loads(metrics[f"unique_labels_{cid}"])

                        for tdict in trees_list:
                            root = SuperTree.Node.from_dict(tdict)
                            tree_nodes.append(root)

            if not tree_nodes:
                return aggregated_metrics

            # 3) fusionar
            st = SuperTree()
            st.mergeDecisionTrees(
                roots=tree_nodes,
                num_classes=len(class_names),
                feature_names=feature_names,
                categorical_features=list(global_mapping.keys()),
                global_mapping=global_mapping,
            )

            # print("\n[SERVIDOR] SuperTree unpruned:")
            # print(st)
            # print("\n")

            # print("\n[SERVIDOR] SuperTree prune_redundant_leaves_full:")
            st.prune_redundant_leaves_full()
            # print(st)
            # print("\n")


            # print("\n[SERVIDOR] SuperTree merge_equal_class_leaves:")
            # st.merge_equal_class_leaves()

            # print(supertree)
            # print("\n")
            
            # print("\n")

            # 4) guardar/emitir
            save_supertree_plot(
                root_node=st.root,
                round_number=server_round,
                feature_names=feature_names,
                class_names=class_names,
                numeric_features=numeric_features,
                global_mapping=global_mapping,   # sin scaler
            )

            LATEST_SUPERTREE_JSON = json.dumps(st.root.to_dict())
            GLOBAL_MAPPING_JSON = json.dumps(global_mapping)
            FEATURE_NAMES_JSON = json.dumps(feature_names)

        except Exception as e:
            print(f"[SERVIDOR] ❌ Error en SuperTree: {e}")

        return aggregated_metrics


    strategy.aggregate_evaluate = custom_aggregate_evaluate
    return ServerAppComponents(strategy=strategy, config=ServerConfig(num_rounds=NUM_SERVER_ROUNDS))

# ============================
# 🧩 FUNCIONES AUXILIARES
# ============================
def _inject_round(original_fn):
    def wrapper(server_round, parameters, client_manager):
        global LATEST_SUPERTREE_JSON, GLOBAL_MAPPING_JSON, FEATURE_NAMES_JSON
        instructions = original_fn(server_round, parameters, client_manager)
        for _, ins in instructions:
            ins.config["server_round"] = server_round

            # Siempre mandamos el último SuperTree disponible
            if LATEST_SUPERTREE_JSON:
                ins.config["supertree"] = LATEST_SUPERTREE_JSON
                ins.config["global_mapping"] = GLOBAL_MAPPING_JSON
                ins.config["feature_names"] = FEATURE_NAMES_JSON

            # Ronda final: modo solo explicación
            if server_round == NUM_SERVER_ROUNDS:
                ins.config["explain_only"] = True
        return instructions
    return wrapper



def print_supertree_legible_fusionado(
    node,
    feature_names,
    class_names,
    numeric_features,
    scaler,  # dict con mean y std
    global_mapping,
    depth=0
):
    import numpy as np
    indent = "|   " * depth
    if node is None:
        print(f"{indent}[Nodo None]")
        return

    if getattr(node, "is_leaf", False):
        class_idx = int(np.argmax(node.labels))
        print(f"{indent}class: {class_names[class_idx]} (pred: {node.labels})")
        return

    feat_idx = node.feat
    feat_name = feature_names[feat_idx]
    intervals = node.intervals
    children = node.children

    # ====== NUMÉRICA ======
    if feat_name in numeric_features:
        bounds = [-np.inf] + list(intervals)
        while len(bounds) < len(children) + 1:
            bounds.append(np.inf)

        for i, child in enumerate(children):
            left = bounds[i]
            right = bounds[i + 1]
            left_real  = left
            right_real = right

            if i == 0:
                cond = f"{feat_name} ≤ {right_real:.2f}"
            elif i == len(children) - 1:
                cond = f"{feat_name} > {left_real:.2f}"
            else:
                cond = f"{feat_name} ∈ ({left_real:.2f}, {right_real:.2f}]"
            print(f"{indent}{cond}")
            print_supertree_legible_fusionado(
                child, feature_names, class_names, numeric_features,
                scaler=None,  # ya no se usa
                global_mapping=global_mapping, depth=depth + 1
            )

    # ====== CATEGÓRICA ONEHOT ======
    elif "=" in feat_name or "_" in feat_name:
        # Soporta 'var=valor' o 'var_valor'
        if "=" in feat_name:
            var, val = feat_name.split("=", 1)
        else:
            var, val = feat_name.split("_", 1)
        var = var.strip()
        val = val.strip()

        if len(children) != 2:
            print(f"[ERROR] Nodo OneHot {feat_name} tiene {len(children)} hijos, esperado 2.")

        # Primero !=, luego ==
        conds = [
            f'{var} != "{val}"',
            f'{var} == "{val}"'
        ]
        for i, child in enumerate(children):
            print(f"{indent}{conds[i]}")
            print_supertree_legible_fusionado(
                child, feature_names, class_names, numeric_features, scaler, global_mapping, depth + 1
            )

    # ====== CATEGÓRICA ORDINAL ======
    elif global_mapping and feat_name in global_mapping:
        vals_cat = global_mapping[feat_name]
        # Primero !=, luego ==
        for i, child in enumerate(children):
            try:
                val_idx = node.intervals[i] if hasattr(node, "intervals") and i < len(node.intervals) else int(getattr(node, "thresh", 0))
                val = vals_cat[val_idx] if val_idx < len(vals_cat) else f"desconocido({val_idx})"
            except Exception as e:
                print(f"[DEPURACIÓN] Error interpretando categórica: {e}")
                val = "?"
            cond = f'{feat_name} != "{val}"' if i == 0 else f'{feat_name} == "{val}"'
            print(f"{indent}{cond}")
            print_supertree_legible_fusionado(
                child, feature_names, class_names, numeric_features, scaler, global_mapping, depth + 1
            )

    # ====== TIPO DESCONOCIDO ======
    else:
        print(f"{indent}{feat_name} [tipo desconocido]")
        print(f"    [DEPURACIÓN] Nombres de features: {feature_names}")
        print(f"    [DEPURACIÓN] Nombres numéricas: {numeric_features}")
        print(f"    [DEPURACIÓN] global_mapping: {list(global_mapping.keys()) if global_mapping else None}")
        print(f"    [DEPURACIÓN] children: {len(children)}")
        for child in children:
            print_supertree_legible_fusionado(
                child, feature_names, class_names, numeric_features, scaler, global_mapping, depth + 1
            )



def save_supertree_plot(
    root_node,
    round_number,
    feature_names,
    class_names,
    numeric_features,
    global_mapping,
    folder="Supertree",
):
    from graphviz import Digraph
    import numpy as np
    import os

    dot = Digraph()
    node_id = [0]

    def add_node(node, parent=None, edge_label=""):
        curr = str(node_id[0]); node_id[0] += 1

        # etiqueta
        if node.is_leaf:
            class_index = int(np.argmax(node.labels))
            label = f"class: {class_names[class_index]}\n{node.labels}"
        else:
            fname = feature_names[node.feat]
            label = fname.split("_", 1)[0] if "_" in fname else fname

        dot.node(curr, label)
        if parent: dot.edge(parent, curr, label=edge_label)

        if not node.is_leaf:
            fname = feature_names[node.feat]
            # OneHot
            if "_" in fname:
                _, val = fname.split("_", 1)
                add_node(node.children[0], curr, f'≠ "{val.strip()}"')
                add_node(node.children[1], curr, f'= "{val.strip()}"')
            # Numérica
            elif fname in numeric_features:
                thr = node.intervals[0] if node.intervals else node.thresh
                add_node(node.children[0], curr, f"≤ {thr:.2f}")
                add_node(node.children[1], curr, f"> {thr:.2f}")
            # Categórica ordinal
            elif fname in global_mapping:
                vals = global_mapping[fname]
                val = vals[node.intervals[0]] if node.intervals else "?"
                add_node(node.children[0], curr, f'= "{val}"')
                add_node(node.children[1], curr, f'≠ "{val}"')
            else:
                for ch in node.children:
                    add_node(ch, curr, "?")

    folder_path = f"Ronda_{round_number}/{folder}"
    os.makedirs(folder_path, exist_ok=True)
    filename = f"{folder_path}/supertree_ronda_{round_number}"
    add_node(root_node)
    dot.render(filename, format="pdf", cleanup=True)
    return f"{filename}.pdf"




# ============================
# 🔧 INICIALIZAR SERVER APP
# ============================
server_app = ServerApp(server_fn=server_fn)



In [ ]:
from flwr.simulation import run_simulation
import logging
import warnings
import ray
import cProfile
import pstats

warnings.filterwarnings("ignore", category=DeprecationWarning)


logging.getLogger('matplotlib').setLevel(logging.WARNING)
logging.getLogger("filelock").setLevel(logging.WARNING)
logging.getLogger("ray").setLevel(logging.WARNING)
logging.getLogger('graphviz').setLevel(logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)  # O ERROR para ocultar aún más
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("fsspec").setLevel(logging.WARNING)
# logging.getLogger("flwr").setLevel(logging.WARNING)




ray.shutdown()  # Apagar cualquier sesión previa de Ray
ray.init(local_mode=True)  # Desactiva multiprocessing, usa un solo proceso principal

backend_config = {"num_cpus": 1}

run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
    backend_config=backend_config,
)

2026-03-24 19:48:42,412	INFO worker.py:1832 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
2026-03-24 19:48:47,976 flwr         DEBUG    Asyncio event loop already running.
:job_id:01000000
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor


:job_id:01000000
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor


INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 2 clients (out of 2)
INFO :      aggregate_fit: received 2 results and 0 failures
INFO :      configure_evaluate: strategy sampled 2 clients (out of 2)


[CLIENTE 1] ✅ LOCAL baseline entrenado y guardado
[CLIENTE 2] ✅ LOCAL baseline entrenado y guardado


INFO :      aggregate_evaluate: received 2 results and 0 failures
INFO :      
INFO :      [ROUND 2]
INFO :      configure_fit: strategy sampled 2 clients (out of 2)


...

INFO :      aggregate_evaluate: received 2 results and 0 failures


Recibiendo supertree....

[SERVIDOR] 🌲 Generando SuperTree - Ronda 2


INFO :      
INFO :      [ROUND 3]
INFO :      configure_fit: strategy sampled 2 clients (out of 2)
INFO :      aggregate_fit: received 2 results and 0 failures
INFO :      configure_evaluate: strategy sampled 2 clients (out of 2)


[CLIENTE 1] 📦 LOCAL baseline recargado en evaluate()
Recibiendo supertree....
[CLIENTE 1] 🔍 Ronda final: solo explicaciones
[CLIENTE 2] 📦 LOCAL baseline recargado en evaluate()
Recibiendo supertree....
[CLIENTE 2] 🔍 Ronda final: solo explicaciones


Cliente 1 explicando test completo:   1%|          | 1/94 [00:28<44:05, 28.44s/it]

### BALANCED METRICS

In [ ]:
# ==========================================
# 📊 Promedio global a partir de los *balanced_mean*
# - Lee results/metrics_cliente_*_balanced_mean.csv con columnas: metric, mean, count
# - Calcula media GLOBAL ponderada por count (no macro-media)
# - Colapsa métricas por clase (TEST y Z) también ponderando por count
# - Guarda en: experimentos_FeatureSkew/<DATASET>/<SKEW_FEATS>/{NCLIENTS}_Clients_Mean_global.csv
# ==========================================
from pathlib import Path
import pandas as pd
import re

# -------------------------------------------------
# ✅ AJUSTA ESTO en tu notebook/entorno:
# - DATASET_NAME (ya lo tienes)
# - SKEW_FEATURE (str o lista de str, ej. ["Age","Sex"])
# -------------------------------------------------

def _slug_feat(name: str) -> str:
    """Normaliza nombres para usarlos como carpeta."""
    name = str(name).strip().replace(" ", "_")
    name = re.sub(r"[^A-Za-z0-9_\-]", "", name)
    return name

def _skew_tag(skew_feature) -> str:
    """Convierte skew_feature (str/list/tuple/set) en 'Age_Sex'."""
    if isinstance(skew_feature, (list, tuple, set)):
        parts = [_slug_feat(x) for x in skew_feature]
    else:
        parts = [_slug_feat(skew_feature)]
    parts = [p for p in parts if p]
    return "_".join(parts) if parts else "UnknownSkew"

def _weighted_mean(subdf: pd.DataFrame, mean_col: str = "mean", count_col: str = "count"):
    """Media ponderada ignorando NaNs."""
    m = subdf[mean_col]
    c = subdf[count_col]
    mask = m.notna() & c.notna() & (c > 0)
    if mask.sum() == 0:
        return float("nan")
    return float((m[mask] * c[mask]).sum() / c[mask].sum())

# -------------------------------------------------
# 📥 Cargar ficheros balanced_mean
# -------------------------------------------------
csv_dir = Path("results")
files = sorted(csv_dir.glob("metrics_cliente_*_balanced_mean.csv"))

if not files:
    raise FileNotFoundError("No encuentro ficheros metrics_cliente_*_balanced_mean.csv en results/")

print("Voy a usar estos ficheros:")
for f in files:
    print("  -", f.name)

n_clients = len(files)
DATASET_NAME = DATASET_NAME.split("/")[-1]

# -------------------------------------------------
# 📁 Carpeta destino
# -------------------------------------------------
out_dir = Path("experimentos_balanced_RandomForest") / DATASET_NAME
out_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# 📄 Leer cada fichero y acumular en formato largo:
# metric | mean | count | client_file
# -------------------------------------------------
dfs_long = []
for f in files:
    df = pd.read_csv(f)

    # Normaliza nombres por si vinieran raros
    df = df.rename(columns={
        df.columns[0]: "metric",
        df.columns[1]: "mean",
    })
    if len(df.columns) >= 3:
        df = df.rename(columns={df.columns[2]: "count"})
    else:
        # Si no hay count, asumimos 1 (macro-media)
        df["count"] = 1.0

    df["client_file"] = f.stem
    dfs_long.append(df[["metric", "mean", "count", "client_file"]])

all_long = pd.concat(dfs_long, ignore_index=True)

# -------------------------------------------------
# ✅ Media global ponderada por count, por métrica
# -------------------------------------------------
means_df = (
    all_long
    .groupby("metric", as_index=False)
    .apply(lambda g: pd.Series({
        "mean": _weighted_mean(g, "mean", "count"),
        "count": float(g["count"].dropna().sum())
    }))
    .reset_index(drop=True)
)

# ==========================================
# 📉 Colapsar clases (TEST y Z) (ponderado)
# - crea nuevas métricas "colapsadas"
# - luego elimina las métricas por-clase originales
# ==========================================
collapse_patterns = {
    # --------- TEST ----------
    "cf_cov_merged_TEST":   r"^cf_cov_merged_[^_]+_TEST$",
    "cf_comp_merged_TEST":  r"^cf_comp_merged_[^_]+_TEST$",
    "cf_prec_merged_TEST":  r"^cf_prec_merged_[^_]+_TEST$",

    "cf_cov_lore_TEST":     r"^cf_cov_lore_[^_]+_TEST$",
    "cf_comp_lore_TEST":    r"^cf_comp_lore_[^_]+_TEST$",
    "cf_prec_lore_TEST":    r"^cf_prec_lore_[^_]+_TEST$",

    "cf_cov_super_TEST":    r"^cf_cov_super_[^_]+_TEST$",
    "cf_comp_super_TEST":   r"^cf_comp_super_[^_]+_TEST$",
    "cf_prec_super_TEST":   r"^cf_prec_super_[^_]+_TEST$",

    "cf_cov_local_local_TEST":  r"^cf_cov_local_local_[^_]+_TEST$",
    "cf_comp_local_local_TEST": r"^cf_comp_local_local_[^_]+_TEST$",
    "cf_prec_local_local_TEST": r"^cf_prec_local_local_[^_]+_TEST$",

    "cf_cov_local_super_TEST":  r"^cf_cov_local_super_[^_]+_TEST$",
    "cf_comp_local_super_TEST": r"^cf_comp_local_super_[^_]+_TEST$",
    "cf_prec_local_super_TEST": r"^cf_prec_local_super_[^_]+_TEST$",

    "cf_cov_localZ_TEST":   r"^cf_cov_localZ_[^_]+_TEST$",
    "cf_comp_localZ_TEST":  r"^cf_comp_localZ_[^_]+_TEST$",
    "cf_prec_localZ_TEST":  r"^cf_prec_localZ_[^_]+_TEST$",

    "cf_cov_superZ_TEST":   r"^cf_cov_superZ_[^_]+_TEST$",
    "cf_comp_superZ_TEST":  r"^cf_comp_superZ_[^_]+_TEST$",
    "cf_prec_superZ_TEST":  r"^cf_prec_superZ_[^_]+_TEST$",

    # --------- Z ----------
    "cf_cov_merged_Z":   r"^cf_cov_merged_[^_]+_Z$",
    "cf_prec_merged_Z":  r"^cf_prec_merged_[^_]+_Z$",

    "cf_cov_lore_Z":     r"^cf_cov_lore_[^_]+_Z$",
    "cf_prec_lore_Z":    r"^cf_prec_lore_[^_]+_Z$",

    "cf_cov_super_Z":    r"^cf_cov_super_[^_]+_Z$",
    "cf_prec_super_Z":   r"^cf_prec_super_[^_]+_Z$",

    "cf_cov_local_local_Z":  r"^cf_cov_local_local_[^_]+_Z$",
    "cf_prec_local_local_Z": r"^cf_prec_local_local_[^_]+_Z$",

    "cf_cov_local_super_Z":  r"^cf_cov_local_super_[^_]+_Z$",
    "cf_prec_local_super_Z": r"^cf_prec_local_super_[^_]+_Z$",

    "cf_cov_localZ_Z":   r"^cf_cov_localZ_[^_]+_Z$",
    "cf_prec_localZ_Z":  r"^cf_prec_localZ_[^_]+_Z$",

    "cf_cov_superZ_Z":   r"^cf_cov_superZ_[^_]+_Z$",
    "cf_prec_superZ_Z":  r"^cf_prec_superZ_[^_]+_Z$",
}

rows_new = []
for new_name, pattern in collapse_patterns.items():
    sub = means_df[means_df["metric"].str.match(pattern)]
    if len(sub) == 0:
        continue
    rows_new.append({
        "metric": new_name,
        "mean": _weighted_mean(sub, "mean", "count"),
        "count": float(sub["count"].dropna().sum())
    })

if rows_new:
    means_df = pd.concat([means_df, pd.DataFrame(rows_new)], ignore_index=True)

# Elimina las métricas por-clase originales (las que acabas de colapsar)
pattern_drop = (
    r"^cf_(cov|prec|comp)_"
    r"(merged|lore|super|local_local|local_super|localZ|superZ)_"
    r"[^_]+_(TEST|Z)$"
)
means_df = means_df[~means_df["metric"].str.match(pattern_drop)].reset_index(drop=True)

# -------------------------------------------------
# 💾 Guardar resultado final (solo metric,mean)
# -------------------------------------------------
out_path = out_dir / f"{n_clients}_Clients_Mean_global.csv"
means_df[["metric", "mean"]].to_csv(out_path, index=False, encoding="utf-8")
print(f"\n✅ Promedios globales (ponderados por count) guardados en: {out_path}")


Voy a usar estos ficheros:
  - metrics_cliente_1_balanced_mean.csv
  - metrics_cliente_2_balanced_mean.csv

✅ Promedios globales (ponderados por count) guardados en: experimentos_balanced_RandomForest\Diabetes\2_Clients_Mean_global.csv
